# Penalty prediction, take 2: season rate instead of match outcome

`penalty_prediction.ipynb` tried to predict, per team per match, whether a penalty would be
awarded — using leak-free rolling stats, XGBoost, referee history, the works. The honest
conclusion there (see that notebook's "Reading the results") was that **no model reliably
beat a flat home/away/league rate at the match level** — the apparent XGBoost edge didn't
survive a fold-by-fold look; it was noise (fold-to-fold std of the log-loss delta was 5-10x
the mean effect).

But a direct check of the data shows the "good teams get more penalties" effect is real and
sizeable *at the season level* — quartile-by-points-per-game, penalties-per-game roughly
doubles from the worst to the best quartile, monotonically, in every league:

| PPG quartile | Premier League | Championship | Superligaen |
|---|---|---|---|
| Q1 (worst) | 0.101 | 0.080 | 0.111 |
| Q2 | 0.119 | 0.072 | 0.139 |
| Q3 | 0.149 | 0.098 | 0.171 |
| Q4 (best) | 0.150 | 0.100 | 0.190 |

That's the motivation for this notebook: **stop trying to predict a single rare binary event
per match, and instead predict a team's season-long penalty *rate*, then spread that rate
across matches.** A season aggregates 30-46 matches, which averages out most of the
match-to-match noise that swamped the per-match model, while still letting team quality
(which changes slowly, season to season) do the predicting.

Approach:
1. Build a team-season panel: total penalties won, games played, points, goal difference.
2. Predict *next* season's penalty rate from *this* season's quality (goal difference, own
   penalty rate), normalized within-league so promoted/relegated teams are comparable —
   walk-forward by season, same held-out seasons as the original notebook.
3. Convert the predicted season rate back into a per-match, per-team probability (splitting
   home/away using each league's own historical split), and evaluate it with the exact same
   metrics as the original notebook, on the same match-level target — so "does this actually
   help" is a fair, apples-to-apples question, not just a story.

In [1]:
import sqlite3
import numpy as np
import pandas as pd

DB_PATH = '../../infra/data/db/fotmob.db'
conn = sqlite3.connect(DB_PATH)
matches = pd.read_sql_query('SELECT * FROM matches', conn)
penalties = pd.read_sql_query('SELECT * FROM penalties', conn)
conn.close()

m = matches.merge(penalties[['match_id', 'home_pens', 'away_pens']], on='match_id', how='left')
m[['home_pens', 'away_pens']] = m[['home_pens', 'away_pens']].fillna(0)
m['match_date'] = pd.to_datetime(m['match_date'])

print(m.shape)
m.head()

(6007, 12)


,match_id,league_id,match_date,home_team,home_goals,away_team,away_goals,season,round,custom_gw,home_pens,away_pens
0,3610280,Premier_League,2022-05-08,9850,0,8654,4,2021-2022,36,NaN,0,1
1,3610003,Premier_League,2021-10-16,8197,4,10260,2,2021-2022,8,NaN,0,0
2,3610146,Premier_League,2022-01-15,9850,2,8668,1,2021-2022,22,NaN,0,0
3,3610238,Premier_League,2022-04-02,8191,0,8456,2,2021-2022,31,NaN,0,0
4,3610054,Premier_League,2021-11-27,9826,1,10252,2,2021-2022,13,NaN,0,0


## Team-season panel

One row per team per season: games played, points, goal difference, total penalties won.
Seasons with fewer than 20 games are dropped (still-in-progress seasons, e.g. 2026-2027).

In [2]:
home = m[['season', 'league_id', 'home_team', 'home_goals', 'away_goals', 'home_pens']].rename(
    columns={'home_team': 'team_id', 'home_goals': 'gf', 'away_goals': 'ga', 'home_pens': 'pens'})
home['pts'] = np.where(home.gf > home.ga, 3, np.where(home.gf == home.ga, 1, 0))
away = m[['season', 'league_id', 'away_team', 'away_goals', 'home_goals', 'away_pens']].rename(
    columns={'away_team': 'team_id', 'away_goals': 'gf', 'home_goals': 'ga', 'away_pens': 'pens'})
away['pts'] = np.where(away.gf > away.ga, 3, np.where(away.gf == away.ga, 1, 0))
team_match = pd.concat([home, away], ignore_index=True)

season_team = team_match.groupby(['league_id', 'season', 'team_id']).agg(
    games=('pts', 'size'), pts=('pts', 'sum'), gf=('gf', 'sum'), ga=('ga', 'sum'), pens=('pens', 'sum'),
).reset_index()
season_team = season_team[season_team.games >= 20].reset_index(drop=True)
season_team['ppg'] = season_team.pts / season_team.games
season_team['gd'] = season_team.gf - season_team.ga
season_team['pens_per_game'] = season_team.pens / season_team.games

print('team-seasons:', len(season_team))
print('overdispersion (var/mean of pens):', (season_team.pens.var() / season_team.pens.mean()).round(2))
season_team.sort_values(['season', 'league_id']).head()

team-seasons: 288
overdispersion (var/mean of pens): 1.41


,league_id,season,team_id,games,pts,gf,ga,pens,ppg,gd,pens_per_game
0,Championship,2020-2021,8119,46,42,44,60,8,0.913043,-16,0.173913
1,Championship,2020-2021,8283,48,79,59,52,8,1.645833,7,0.166667
2,Championship,2020-2021,8344,46,68,66,49,8,1.478261,17,0.173913
3,Championship,2020-2021,8346,46,62,41,52,5,1.347826,-11,0.108696
4,Championship,2020-2021,8411,46,61,49,56,8,1.326087,-7,0.173913


## Prior-season predictors, normalized within league

For each team-season, look up that *same team's* previous season — whichever league it was
in. To make a promoted team's Championship form comparable to a Premier League incumbent's,
z-score goal difference and points-per-game **within that prior season's own league**, rather
than using raw values (a simpler stand-in for the original notebook's swap-seeding — at
season grain, a single well-normalized number per team does the job without needing to
borrow another team's match history).

`prior_known` flags rows with no lookup at all (a team's very first season in the dataset, or
Superligaen's first season here) — those get the league's own historical mean as a neutral
fallback, the same "shrink fully to the prior when there's nothing to go on" pattern used for
the referee feature and the promotion swap-map in the original notebook.

This is deliberately simpler than a cross-league Elo-based prior would be (ClubElo ratings
are already in this DB, `clubelo_ratings`, and would be a principled upgrade here) — no
name-mapping between FotMob and ClubElo team names exists yet, so that's left as a follow-up
rather than solved in this first pass.

In [3]:
season_team['gd_z'] = season_team.groupby(['league_id', 'season'])['gd'].transform(
    lambda s: (s - s.mean()) / s.std(ddof=0) if s.std(ddof=0) > 0 else 0.0)
season_team['ppg_z'] = season_team.groupby(['league_id', 'season'])['ppg'].transform(
    lambda s: (s - s.mean()) / s.std(ddof=0) if s.std(ddof=0) > 0 else 0.0)

seasons_sorted = sorted(season_team.season.unique(), key=lambda s: int(s.split('-')[0]))
season_index = {s: i for i, s in enumerate(seasons_sorted)}
season_team['season_idx'] = season_team.season.map(season_index)

# Prior-season lookup: same team_id, season_idx - 1, regardless of league.
prior_cols = ['team_id', 'season_idx', 'league_id', 'gd_z', 'ppg_z', 'pens_per_game']
prior = season_team[prior_cols].rename(columns={
    'league_id': 'prior_league', 'gd_z': 'prior_gd_z', 'ppg_z': 'prior_ppg_z', 'pens_per_game': 'prior_pens_pg',
})
prior['season_idx'] = prior['season_idx'] + 1  # shift forward one season so it joins onto the *next* season's row

season_team = season_team.merge(prior, on=['team_id', 'season_idx'], how='left')
season_team['prior_known'] = season_team['prior_gd_z'].notna().astype(int)
season_team['promoted_or_relegated'] = (
    season_team['prior_league'].notna() & (season_team['prior_league'] != season_team['league_id'])
).astype(int)

# Fallback for cold-start rows: neutral z-score, and this league's own historical mean pens/game
# up to (not including) the current season -- no lookahead.
league_hist_mean = {}
for (lg, s_idx), _ in season_team.groupby(['league_id', 'season_idx']):
    hist = season_team[(season_team.league_id == lg) & (season_team.season_idx < s_idx)]
    league_hist_mean[(lg, s_idx)] = hist.pens_per_game.mean() if len(hist) else season_team.pens_per_game.mean()

season_team['league_hist_mean_pens_pg'] = season_team.apply(
    lambda r: league_hist_mean[(r.league_id, r.season_idx)], axis=1)
season_team['prior_gd_z'] = season_team['prior_gd_z'].fillna(0.0)
season_team['prior_ppg_z'] = season_team['prior_ppg_z'].fillna(0.0)
season_team['prior_pens_pg'] = season_team['prior_pens_pg'].fillna(season_team['league_hist_mean_pens_pg'])

print('prior_known rate:', season_team.prior_known.mean().round(3))
print('promoted_or_relegated rate (of rows with a prior):',
      season_team.loc[season_team.prior_known == 1, 'promoted_or_relegated'].mean().round(3))
season_team[['league_id', 'season', 'team_id', 'pens_per_game', 'prior_gd_z', 'prior_ppg_z',
             'prior_pens_pg', 'prior_known', 'promoted_or_relegated']].head(8)

prior_known rate: 0.747
promoted_or_relegated rate (of rows with a prior): 0.14


,league_id,season,team_id,pens_per_game,prior_gd_z,prior_ppg_z,prior_pens_pg,prior_known,promoted_or_relegated
0,Championship,2020-2021,8119,0.173913,0.0,0.0,0.114127,0,0
1,Championship,2020-2021,8283,0.166667,0.0,0.0,0.114127,0,0
2,Championship,2020-2021,8344,0.173913,0.0,0.0,0.114127,0,0
3,Championship,2020-2021,8346,0.108696,0.0,0.0,0.114127,0,0
4,Championship,2020-2021,8411,0.173913,0.0,0.0,0.114127,0,0
5,Championship,2020-2021,8427,0.043478,0.0,0.0,0.114127,0,0
6,Championship,2020-2021,8549,0.065217,0.0,0.0,0.114127,0,0
7,Championship,2020-2021,8655,0.173913,0.0,0.0,0.114127,0,0


## Season-level model: Negative Binomial, walk-forward by season

Same held-out seasons as the original notebook (2022-2023 through 2025-2026), so the two
notebooks' match-level numbers can be compared directly later. Two models per fold:
- `baseline_league_mean` — flat per-league mean penalty rate from the training seasons (the
  season-level equivalent of `baseline_home_away_league`).
- `poisson` — Poisson GLM: `pens ~ prior_gd_z + prior_ppg_z + prior_pens_pg + league +
  promoted_or_relegated`, with an offset of `log(games)` so it models a per-game rate rather
  than a raw count that depends on how many games were played. (Counts are only mildly
  overdispersed here — var/mean = 1.41 — and a Negative Binomial fit was numerically unstable
  on the smallest training folds; Poisson gives the same point predictions a quasi-Poisson
  correction would, which is all that's needed for prediction rather than inference.)

In [4]:
import statsmodels.api as sm
import statsmodels.formula.api as smf

FOLDS_SEASON = [
    (['2021-2022'], '2022-2023'),
    (['2021-2022', '2022-2023'], '2023-2024'),
    (['2021-2022', '2022-2023', '2023-2024'], '2024-2025'),
    (['2021-2022', '2022-2023', '2023-2024', '2024-2025'], '2025-2026'),
]

modelable = season_team[season_team.prior_known == 1].copy()
modelable['league_id'] = modelable['league_id'].astype('category')

FORMULA = 'pens ~ prior_gd_z + prior_ppg_z + prior_pens_pg + C(league_id) + promoted_or_relegated'

season_results = []
season_fold_predictions = {}  # fold_i -> df with season-rate predictions, for the match-level step later
for fold_i, (train_seasons, test_season) in enumerate(FOLDS_SEASON):
    train = modelable[modelable.season.isin(train_seasons)]
    test = modelable[modelable.season == test_season]
    if len(train) < 20 or len(test) < 5:
        continue

    # Baseline: flat per-league mean rate from training data only.
    league_mean_rate = train.groupby('league_id', observed=True)['pens_per_game'].mean()
    base_pred = test['league_id'].map(league_mean_rate).fillna(train['pens_per_game'].mean()).to_numpy()

    # Poisson GLM with an exposure offset for games played. (Counts are only mildly
    # overdispersed -- var/mean 1.41 -- and a Negative Binomial fit was numerically unstable
    # on the smallest training folds here (singular Hessian, too few effective d.o.f. for the
    # extra dispersion parameter). Poisson gives the same point predictions a quasi-Poisson
    # would; we only need the predicted rate here, not its standard error.)
    model = smf.glm(FORMULA, data=train, family=sm.families.Poisson(), offset=np.log(train['games'])).fit()
    nb_pred = model.predict(test, offset=np.log(test['games'])) / test['games']

    test_out = test[['league_id', 'season', 'team_id', 'games', 'pens_per_game']].copy()
    test_out['pred_league_mean'] = base_pred
    test_out['pred_negbin'] = nb_pred.to_numpy()
    season_fold_predictions[fold_i] = test_out

    for label, pred in [('baseline_league_mean', base_pred), ('poisson', nb_pred.to_numpy())]:
        # Poisson deviance on the per-game rate -- proper scoring rule for a rate/count target.
        eps = 1e-6
        actual = test['pens_per_game'].to_numpy()
        pred_c = np.clip(pred, eps, None)
        actual_c = np.clip(actual, eps, None)
        deviance = 2 * np.mean(actual_c * np.log(actual_c / pred_c) - (actual_c - pred_c))
        mae = np.mean(np.abs(actual - pred))
        season_results.append({
            'fold': fold_i, 'test_season': test_season, 'model': label,
            'n_test': len(test), 'poisson_deviance': deviance, 'mae': mae,
        })

season_results_df = pd.DataFrame(season_results)
season_results_df.pivot(index='test_season', columns='model', values=['poisson_deviance', 'mae']).round(4)

poisson_deviance                          mae        
model       baseline_league_mean poisson baseline_league_mean poisson
test_season                                                          
2022-2023                 0.0297  0.0273               0.0469  0.0447
2023-2024                 0.0438  0.0420               0.0583  0.0577
2024-2025                 0.0284  0.0284               0.0427  0.0429
2025-2026                 0.0418  0.0450               0.0501  0.0524

In [5]:
season_results_df.groupby('model')[['poisson_deviance', 'mae']].mean().round(4)

,poisson_deviance,mae
model,,
baseline_league_mean,0.0359,0.0495
poisson,0.0357,0.0494


## Converting the season rate to a per-match probability

Each team-season row now has a predicted *average* penalties-per-game rate. To turn that into
a per-match, per-team-per-side probability, split it into home/away components using each
league's own empirical home/away split — learned from **training data only**, same
no-lookahead discipline as everywhere else in this project — then treat a single match as a
Poisson trial: `P(pen this match) = 1 - exp(-λ)`.

This produces a number in exactly the same units as the original notebook's target (`did this
team get awarded a penalty in this match`), so it can be scored with the same log-loss/Brier/
AUC and compared directly to that notebook's numbers:
- `baseline_home_away_league` (original): log-loss ≈ 0.3229, Brier ≈ 0.0897
- `xgboost` (original, not distinguishable from the baseline once checked fold-by-fold): log-loss ≈ 0.3227, Brier ≈ 0.0896

In [6]:
from sklearn.metrics import log_loss, brier_score_loss, roc_auc_score

# Rebuild the match-level, per-team-per-match target exactly as the original notebook did.
home_rows = m[['match_id', 'match_date', 'league_id', 'season', 'home_team', 'home_pens']].rename(
    columns={'home_team': 'team_id', 'home_pens': 'pens_awarded'})
home_rows['is_home'] = 1
away_rows = m[['match_id', 'match_date', 'league_id', 'season', 'away_team', 'away_pens']].rename(
    columns={'away_team': 'team_id', 'away_pens': 'pens_awarded'})
away_rows['is_home'] = 0
match_panel = pd.concat([home_rows, away_rows], ignore_index=True)
match_panel['target'] = (match_panel['pens_awarded'] > 0).astype(int)

match_level_results = []
for fold_i, (train_seasons, test_season) in enumerate(FOLDS_SEASON):
    if fold_i not in season_fold_predictions:
        continue
    train_matches = match_panel[match_panel.season.isin(train_seasons)]

    # Home/away split per league, from training matches only.
    home_share = train_matches[train_matches.is_home == 1].groupby('league_id')['target'].mean()
    away_share = train_matches[train_matches.is_home == 0].groupby('league_id')['target'].mean()
    home_mult = (2 * home_share / (home_share + away_share)).to_dict()
    away_mult = (2 * away_share / (home_share + away_share)).to_dict()
    # Fallback for a league with no training history at all yet (e.g. Superligaen's first
    # test season, 2024-2025, trained only on PL/Championship seasons before it): use the
    # pooled overall home/away split rather than leaving a NaN multiplier.
    overall_home_share = train_matches.loc[train_matches.is_home == 1, 'target'].mean()
    overall_away_share = train_matches.loc[train_matches.is_home == 0, 'target'].mean()
    overall_home_mult = 2 * overall_home_share / (overall_home_share + overall_away_share)
    overall_away_mult = 2 * overall_away_share / (overall_home_share + overall_away_share)

    preds = season_fold_predictions[fold_i][['league_id', 'season', 'team_id', 'pred_negbin']]
    test_matches = match_panel[match_panel.season == test_season].merge(preds, on=['league_id', 'season', 'team_id'], how='left')
    # Cold-start teams with no season-level prediction (shouldn't normally happen for a fully
    # in-progress test season, but guard anyway): fall back to the league's flat training rate.
    fallback_rate = train_matches.groupby('league_id')['target'].mean()
    overall_fallback_rate = train_matches['target'].mean()
    test_matches['pred_negbin'] = test_matches['pred_negbin'].fillna(
        test_matches['league_id'].map(fallback_rate).fillna(overall_fallback_rate))

    home_mult_map = test_matches['league_id'].map(home_mult).fillna(overall_home_mult)
    away_mult_map = test_matches['league_id'].map(away_mult).fillna(overall_away_mult)
    lam = np.where(
        test_matches['is_home'] == 1,
        test_matches['pred_negbin'] * home_mult_map,
        test_matches['pred_negbin'] * away_mult_map,
    )
    season_rate_proba = 1 - np.exp(-lam)

    # Same-shape league-split baseline as the original notebook, for reference.
    rate_table = train_matches.groupby(['league_id', 'is_home'])['target'].mean()
    baseline_proba = np.array([
        rate_table.get((lg, h), train_matches['target'].mean())
        for lg, h in zip(test_matches['league_id'], test_matches['is_home'])
    ])

    y_test = test_matches['target']
    for label, proba in [('baseline_home_away_league', baseline_proba), ('season_rate_negbin', season_rate_proba)]:
        match_level_results.append({
            'fold': fold_i, 'test_season': test_season, 'model': label, 'n_test': len(y_test),
            'log_loss': log_loss(y_test, proba, labels=[0, 1]),
            'brier': brier_score_loss(y_test, proba),
            'auc': roc_auc_score(y_test, proba),
        })

match_level_df = pd.DataFrame(match_level_results)
print(match_level_df.pivot(index='test_season', columns='model', values='log_loss').round(4))
print()
match_level_df.groupby('model')[['log_loss', 'brier', 'auc']].mean().round(4)

model        baseline_home_away_league  season_rate_negbin
test_season                                               
2022-2023                       0.3248              0.3236
2023-2024                       0.3280              0.3274
2024-2025                       0.3130              0.3135
2025-2026                       0.3181              0.3204



,log_loss,brier,auc
model,,,
baseline_home_away_league,0.3210,0.0892,0.5826
season_rate_negbin,0.3212,0.0892,0.5785


## Reading the results

**The season-rate reframing does not beat the flat baseline either — but for a different,
more interesting reason than the per-match model's failure.**

At the season level, the Poisson model (prior-season goal difference, points, and own penalty
rate) is essentially tied with a flat per-league mean: mean Poisson deviance 0.0357 vs 0.0359,
mean MAE 0.0494 vs 0.0495 — and the per-fold picture is mixed (better in 2022-2023 and
2023-2024, tied in 2024-2025, worse in 2025-2026), the same noisy, no-consistent-winner
pattern that sank the per-match XGBoost model.

That's a genuinely different failure mode than before, though. The direct, same-season check
at the top of this notebook showed a strong, real relationship between team quality and
penalty rate (quartile 4 teams draw ~25-70% more penalties per game than quartile 1, in every
league). The season model uses *last* season's quality to predict *this* season's rate — and
that one-year gap is where the signal mostly evaporates. Team quality itself doesn't persist
that cleanly year to year (transfers, managerial changes, squad turnover), so "how good this
team was last year" is a much noisier proxy for "how good this team is now" than it first
appears. The match-level comparison confirms the same wash: log-loss 0.3212 vs 0.3210 for the
baseline, again flipping sign fold to fold (better in 2022-2023/2023-2024, worse in
2024-2025/2025-2026).

**So the honest three-part version of "penalties are hard to predict" is:**
1. At the match level, team-form signal is real (weak correlations, ~0.05-0.08) but too small
   relative to match-to-match noise for any model tested to reliably beat a flat rate.
2. Aggregated to a full season using *contemporaneous* data, the team-quality effect is
   large and unambiguous (quartile 1 vs quartile 4 roughly doubles).
3. But forecasting that effect *forward* a full season using only the *prior* season's
   quality mostly loses it again — not because of match-level noise this time, but because
   team quality itself doesn't persist across the close-season well enough for a
   year-old snapshot to predict it.

### What this suggests trying next
The gap in (3) points at the real next experiment: don't rely on a full season-old snapshot.
Use an **in-season, continuously-updated** rate instead — blend the prior-season-derived
expectation with the *current* season's own accumulating penalty count as it plays out (a
Negative-Binomial/Gamma conjugate update, shrinking hard toward the prior in August and
toward the observed in-season rate by March). That directly targets the actual weak point
found here (a stale, one-year-old quality signal) rather than either extreme already tried
(match-level noise on one end, season-old staleness on the other) — a "nowcast" sitting
between the two failed approaches instead of at either end.

Other upgrades, in priority order:
- **In-season Bayesian updating** (above) — the most direct fix for what actually failed here.
- **Cross-league Elo prior** (`clubelo_ratings` is already collected) would replace the
  within-league z-score normalization with a properly cross-league-comparable quality signal
  — needs a FotMob-to-ClubElo team name mapping, which doesn't exist yet.
- **More seasons** would help distinguish whether the season-to-season persistence gap here
  is a real, stable property of team quality, or itself partly a small-sample artifact (only
  4 test seasons, 3-4 training seasons per fold) — same caveat as the per-match notebook.

## In-season Bayesian updating

The gap found above: a preseason (prior-season-derived) rate is a weak forecast of this
season's actual rate, because team quality doesn't persist cleanly across the close-season.
The fix that directly targets that gap — rather than either extreme already tried (full
season-old snapshot, or noisy per-match features) — is a **nowcast**: start each team-season
at its preseason-predicted rate, then blend in that team's own *this-season* observed penalty
count as it accumulates, shrinking away from the (weak) preseason prior and toward the
in-season observed rate as more of the season is played.

This is a Poisson-Gamma conjugate update. Treat a team's true per-game penalty rate as
Gamma-distributed; a Gamma(`k * prior_rate`, `k`) prior updated with `c` penalties observed
over `g` games so far this season gives a posterior mean of:

```
blended_rate = (k * prior_rate + c) / (k + g)
```

`k` is "pseudo-games" of trust placed in the preseason prior — small `k` means the model
trusts a handful of this-season matches enough to override the prior quickly; large `k` means
it takes most of a season to move away from the preseason number. `c` and `g` only ever count
matches strictly before the one being predicted (no lookahead), same discipline as everywhere
else in this project. A few values of `k` are tried below rather than picked by guesswork —
same instinct as the window-selection question earlier: let the walk-forward folds show
which value actually helps, and look at the per-fold detail before trusting the average.

In [7]:
# Cumulative in-season games/pens per team, strictly before each match (no lookahead).
team_season_matches = match_panel.sort_values(['team_id', 'season', 'match_date']).copy()
team_season_matches['games_so_far'] = team_season_matches.groupby(['team_id', 'season']).cumcount()
team_season_matches['pens_so_far'] = (
    team_season_matches.groupby(['team_id', 'season'])['target'].cumsum() - team_season_matches['target']
)

K_VALUES = [5, 10, 20, 30, 50]

bayes_results = []
for fold_i, (train_seasons, test_season) in enumerate(FOLDS_SEASON):
    if fold_i not in season_fold_predictions:
        continue
    train_matches = match_panel[match_panel.season.isin(train_seasons)]

    home_share = train_matches[train_matches.is_home == 1].groupby('league_id')['target'].mean()
    away_share = train_matches[train_matches.is_home == 0].groupby('league_id')['target'].mean()
    home_mult = (2 * home_share / (home_share + away_share)).to_dict()
    away_mult = (2 * away_share / (home_share + away_share)).to_dict()
    overall_home_share = train_matches.loc[train_matches.is_home == 1, 'target'].mean()
    overall_away_share = train_matches.loc[train_matches.is_home == 0, 'target'].mean()
    overall_home_mult = 2 * overall_home_share / (overall_home_share + overall_away_share)
    overall_away_mult = 2 * overall_away_share / (overall_home_share + overall_away_share)

    preds = season_fold_predictions[fold_i][['league_id', 'season', 'team_id', 'pred_negbin']]
    test_matches = team_season_matches[team_season_matches.season == test_season].merge(
        preds, on=['league_id', 'season', 'team_id'], how='left')
    fallback_rate = train_matches.groupby('league_id')['target'].mean()
    overall_fallback_rate = train_matches['target'].mean()
    test_matches['pred_negbin'] = test_matches['pred_negbin'].fillna(
        test_matches['league_id'].map(fallback_rate).fillna(overall_fallback_rate))

    home_mult_map = test_matches['league_id'].map(home_mult).fillna(overall_home_mult)
    away_mult_map = test_matches['league_id'].map(away_mult).fillna(overall_away_mult)
    mult_map = np.where(test_matches['is_home'] == 1, home_mult_map, away_mult_map)

    y_test = test_matches['target']
    for k in K_VALUES:
        blended_rate = (k * test_matches['pred_negbin'] + test_matches['pens_so_far']) / (k + test_matches['games_so_far'])
        lam = blended_rate * mult_map
        proba = 1 - np.exp(-lam)
        bayes_results.append({
            'fold': fold_i, 'test_season': test_season, 'model': f'bayes_k{k}', 'n_test': len(y_test),
            'log_loss': log_loss(y_test, proba, labels=[0, 1]),
            'brier': brier_score_loss(y_test, proba),
            'auc': roc_auc_score(y_test, proba),
        })

bayes_results_df = pd.DataFrame(bayes_results)
compare_all = pd.concat([match_level_df, bayes_results_df], ignore_index=True)
compare_all.groupby('model')[['log_loss', 'brier', 'auc']].mean().round(4).sort_values('log_loss')

,log_loss,brier,auc
model,,,
bayes_k50,0.3205,0.0891,0.5859
baseline_home_away_league,0.3210,0.0892,0.5826
bayes_k30,0.3212,0.0892,0.5812
season_rate_negbin,0.3212,0.0892,0.5785
bayes_k20,0.3223,0.0894,0.5765
bayes_k10,0.3258,0.0899,0.5677
bayes_k5,0.3315,0.0907,0.5601


In [8]:
# Per-fold detail for the best-looking k, to check it's not another window-25-style mirage.
best_k_model = bayes_results_df.groupby('model')['log_loss'].mean().idxmin()
print('Best k by mean log-loss:', best_k_model)
detail = compare_all[compare_all.model.isin(['baseline_home_away_league', 'season_rate_negbin', best_k_model])]
detail.pivot(index='test_season', columns='model', values='log_loss').round(4)

Best k by mean log-loss: bayes_k50


model,baseline_home_away_league,bayes_k50,season_rate_negbin
test_season,,,
2022-2023,0.3248,0.3248,0.3236
2023-2024,0.3280,0.3246,0.3274
2024-2025,0.3130,0.3136,0.3135
2025-2026,0.3181,0.3191,0.3204


### Reading the in-season updating result

**Same pattern as everything else in these two notebooks: no reliable win, and a clear reason
why not.**

`k=50` (heavy shrinkage — a team needs about 50 games' worth of its own evidence to fully
override the preseason number, more than a full season) gets the best mean log-loss (0.3205
vs baseline 0.3210), but the per-fold detail is the familiar story: a real win in 2023-2024
(0.3246 vs 0.3280), a wash in 2022-2023, and small losses in 2024-2025 and 2025-2026. One good
fold pulling the average down, not a consistent effect.

More telling is what happens at *lower* `k`: performance gets steadily **worse** as the model
trusts in-season evidence more (`k=5` is the worst of everything tried, log-loss 0.3315).
That's the mechanism, not just a tuning failure — a team's own in-season pens-so-far count is
itself a terrible estimator early in a season. At `pens_per_game` around 0.1-0.2, a team is
lucky to have 2-4 penalty events in its first 20 games; blending that noisy handful of events
in with real weight makes predictions *worse*, not better, exactly like reading too much into
4 walk-forward folds. Only shrinking so hard that the blend barely moves from the (already weak) preseason
prior avoids doing damage — which means this version of "nowcasting" isn't actually adding
information, just approximately reproducing the preseason model with extra steps.

**Overall conclusion across both notebooks**: every angle tried — rich per-match ML features,
season-ahead forecasting from prior-season quality, and in-season Bayesian updating — either
fails to beat a flat home/away/league rate, or "beats" it by a margin indistinguishable from
the fold-to-fold noise once checked honestly. The rare-event, high-variance nature of
penalties, combined with team quality not persisting cleanly season to season, appears to be
a real ceiling with this dataset and these features — not a modeling failure to fix with a
better algorithm. The two most promising remaining levers are structural, not algorithmic:
**more seasons of data** (weak-but-real effects need much bigger samples to separate from
noise) and **a genuinely new source of signal** (referee identity showed a modest, real
correlation on its own terms in the first notebook; shot-location detail and a cross-league
Elo prior are the other untried candidates) — not further tuning of how the existing signal
gets aggregated or blended.